In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neural_network import MLPClassifier

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

In [ ]:
# --- 1. Load Data ---
# We will load the dataset directly from the UCI repository URL instead of a local file path
# to ensure it runs anywhere (like Colab or local environemnts without manual download).

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip"
print(f"Downloading and loading data from {url}...")

# Pandas can read inside zip files directly if we specify the file name
df = pd.read_csv(url, compression='zip', sep=';')

# If the above direct zip read fails (sometimes creating issues with nested folders inside zip),
# we fall back to a direct CSV link if available, or assume local file if user provides it.
# For now, let's try reading the 'bank-additional-full.csv' which is usually inside.
# Note: The zip contains a folder structure. We might need to handle that.

try:
    # Try reading directly assuming flat structure or smart pandas handling
    # actually read_csv with zip compression expects a single file. 
    # Let's use a cleaner approach: downloading and extracting if needed, 
    # OR just use a mirror that serves the raw CSV. 
    # Let's use a popular mirror for this specific CSV to be safe and simple.
    url_csv = "https://raw.githubusercontent.com/fivethirtyeight/data/master/bank-marketing/bank-additional-full.csv"
    # Wait, 538 might not have the full additional one. 
    # Let's stick to the reliable method: using `requests` and `zipfile` if strictly needed, 
    # but for a notebook, let's try to assume the user might have it OR use a workable direct link.
    # A reliable source for the raw CSV:
    url_direct = "https://media.githubusercontent.com/media/Kirans1ngh/Machine-Learning-practice/refs/heads/main/Classification/bank-additional-full.csv" # Hypothesis: It might not be there yet.
    
    # Let's use the code the user provided but adapted to be robust.
    # User code: df = pd.read_csv('/content/bank-additional-full.csv', sep=';')
    
    # We will try to read from a reliable public S3 bucket or similar if possible,
    # otherwise we'll instruct user to place the file.
    # Actually, let's use the UCI remote directly with specific file selection if pandas supports it? No.
    
    # Let's just use a very standard public URL for this famous dataset.
    df = pd.read_csv('https://raw.githubusercontent.com/uci-ml-repo/bank-marketing/main/bank-additional/bank-additional-full.csv', sep=';')
except:
    # Fallback to the one commonly used in tutorials
    print("Could not download directly from GitHub mirror. Trying an alternative...")
    # If this fails, we will create a dummy dataset or ask user to provide file.
    # SAFE BET: The user seems to have run it in Colab. 
    # I will write the code to expect the file in the current directory or download it.
    import urllib.request
    import zipfile
    import os
    
    if not os.path.exists('bank-additional-full.csv'):
        print("Downloading dataset...")
        urllib.request.urlretrieve("https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip", "bank-additional.zip")
        with zipfile.ZipFile("bank-additional.zip", 'r') as zip_ref:
            zip_ref.extractall(".")
        # It extracts to a folder 'bank-additional'. Move file out or read from there.
        df = pd.read_csv('bank-additional/bank-additional-full.csv', sep=';')
    else:
        df = pd.read_csv('bank-additional-full.csv', sep=';')

print("Data loaded successfully.")
df.head()

In [ ]:
# --- 2. Preprocessing ---

# Convert target to 0/1
df['y'] = df['y'].map({'yes': 1, 'no': 0})

# Features & Target
X = df.drop('y', axis=1)
y = df['y']

# Identify column types
num_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

print(f"Numerical columns: {len(num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")

# Pipelines
num_pipeline = Pipeline([('scaler', StandardScaler())])
cat_pipeline = Pipeline([('encoder', OneHotEncoder(handle_unknown='ignore'))])

preprocess = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# --- 3. Model: Naive Bayes ---

print("Training Naive Bayes...")
nb_model = Pipeline([
    ('prep', preprocess),
    ('clf', GaussianNB())
])
nb_model.fit(X_train, y_train)
pred_nb = nb_model.predict(X_test)

print("\n🔥 NAIVE BAYES RESULTS 🔥")
print("Accuracy:", accuracy_score(y_test, pred_nb))
print(classification_report(y_test, pred_nb))

# Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, pred_nb, cmap='Blues')
plt.title("Naive Bayes Confusion Matrix")
plt.show()

In [ ]:
# --- 4. Model: Decision Tree ---

print("Training Decision Tree...")
dt_model = Pipeline([
    ('prep', preprocess),
    ('clf', DecisionTreeClassifier(random_state=42))
])
dt_model.fit(X_train, y_train)
pred_dt = dt_model.predict(X_test)

print("\n🌲 DECISION TREE RESULTS 🌲")
print("Accuracy:", accuracy_score(y_test, pred_dt))
print(classification_report(y_test, pred_dt))

# Visualizing the tree (limited depth for readability)
plt.figure(figsize=(20,10))
# Note: plotting pipeline steps can be tricky. We access the classifier step.
# And getting feature names from OneHotEncoder is verbose, so we skip detailed labels for this large tree.
plot_tree(dt_model.named_steps['clf'], max_depth=3, filled=True, fontsize=10)
plt.title("Decision Tree (Max Depth 3 view)")
plt.show()

In [ ]:
# --- 5. Model: Neural Network (MLP) ---

print("Training MLP Classifier (this may take a few seconds)...")
mlp_model = Pipeline([
    ('prep', preprocess),
    ('clf', MLPClassifier(
        hidden_layer_sizes=(100,),
        activation='relu',
        solver='adam',
        max_iter=300,
        random_state=42
    ))
])
mlp_model.fit(X_train, y_train)
pred_mlp = mlp_model.predict(X_test)

print("\n🤖 NEURAL NETWORK (MLP) RESULTS 🤖")
print("Accuracy:", accuracy_score(y_test, pred_mlp))
print(classification_report(y_test, pred_mlp))

In [ ]:
# --- 6. Comparison ---

results = pd.DataFrame({
    'Model': ['Naive Bayes', 'Decision Tree', 'MLP Classifier'],
    'Accuracy': [
        accuracy_score(y_test, pred_nb),
        accuracy_score(y_test, pred_dt),
        accuracy_score(y_test, pred_mlp)
    ]
})

print(results)
sns.barplot(x='Model', y='Accuracy', data=results, palette='viridis')
plt.title("Model Accuracy Comparison")
plt.ylim(0.7, 1.0)
plt.show()